In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import requests
from src.pipeline.clean_datas import clean_datas_history

#from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    auc,
    ConfusionMatrixDisplay
)

In [2]:
df_train = pd.read_csv("../datas/fraudTrain.csv")

df_clean_train = clean_datas_history(df_train)
df_clean_train.sample(10)

,category,amt,gender,state,city_pop,is_fraud,trans_hour,trans_day,trans_month,age,distance_km,customer_job_category
877838,personal_care,4.02,F,MT,743,0,22,21,12,54,89,tech
852750,entertainment,5.19,F,NY,1166,0,22,15,12,43,60,tech
606820,gas_transport,59.80,M,MS,2870,0,3,15,9,27,63,finance
487528,personal_care,7.34,M,OH,272134,0,12,1,8,62,44,management
243340,gas_transport,73.23,M,KY,564,0,4,2,5,30,123,finance
601104,home,51.21,M,MI,134056,0,16,12,9,70,106,creative
142979,kids_pets,15.05,F,CA,1661,0,12,18,3,38,82,health
986424,shopping_net,150.70,M,NC,5354,0,4,6,2,61,106,commerce
371305,grocery_pos,111.29,M,WV,571,0,6,21,6,59,23,tech
1121509,misc_net,7.50,M,HI,4878,0,9,12,4,60,55,health


In [3]:
df_clean_train.shape

(1296675, 12)

In [4]:
df_test = pd.read_csv("../datas/fraudTest.csv")

df_clean_test = clean_datas_history(df_test)
df_clean_test.sample(10)

,category,amt,gender,state,city_pop,is_fraud,trans_hour,trans_day,trans_month,age,distance_km,customer_job_category
444536,misc_pos,7.43,M,SC,20478,0,13,7,12,29,56,tech
115941,grocery_net,43.81,M,WI,1360,0,0,1,8,42,90,health
241920,personal_care,46.87,M,IN,137,0,18,16,9,59,86,tech
153062,entertainment,24.19,F,WV,836,0,20,13,8,45,54,health
92779,health_fitness,60.10,F,CT,647,0,13,23,7,39,9,management
110223,grocery_pos,127.64,F,SC,1523,0,7,29,7,42,89,science
132636,home,56.35,F,SC,4913,0,15,6,8,60,88,tech
446142,health_fitness,211.52,F,OH,479994,0,18,7,12,56,107,management
377363,health_fitness,75.35,F,FL,3699,0,17,15,11,78,35,management
53559,shopping_pos,29.42,F,AZ,2872,0,19,9,7,39,60,tech


In [5]:
df_clean_test.shape

(555719, 12)

In [6]:
feature_target = "is_fraud"

X_test = df_clean_test.drop(columns=[feature_target])
y_test = df_clean_test[feature_target]

X_train = df_clean_train.drop(columns=[feature_target])
y_train = df_clean_train[feature_target]

In [7]:
numeric_features = []
categorical_features = []

for i,t in X_train.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)
        
print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['amt', 'city_pop', 'trans_hour', 'trans_day', 'trans_month', 'age', 'distance_km']
Found categorical features  ['category', 'gender', 'state', 'customer_job_category']


In [8]:
numeric_transformer = Pipeline(
    steps=[
        ('scaler', StandardScaler())
    ])

categorical_transformer = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [9]:
# models = {
#     "Logistic Regression": LogisticRegression(),
#     "Random Forest": RandomForestClassifier(),
#     "XGBoost": XGBClassifier(eval_metric="logloss"),
# }

# results = []

# for name, model in models.items():
#     pipeline = Pipeline(steps=[
#         ('preprocessor', preprocessor),
#         ('classifier', model)
#     ])
#     scores = cross_val_score(pipeline, X_train, y_train, cv=10)
#     results.append({
#         "Model": name,
#         "Mean Accuracy": scores.mean(),
#         "Standard Deviation": scores.std()
#     })

# results_df = pd.DataFrame(results)
# results_df.sort_values(by="Mean Accuracy", ascending=False)

In [14]:
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight="balanced", max_iter=1000))
])

pipeline_lr.fit(X_train, y_train)

# Prédiction
y_pred = pipeline_lr.predict(X_test)

# Évaluation
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.92      0.96    553574
           1       0.03      0.73      0.06      2145

    accuracy                           0.92    555719
   macro avg       0.52      0.82      0.51    555719
weighted avg       1.00      0.92      0.95    555719



In [15]:
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight="balanced"))
])

pipeline_rf.fit(X_train, y_train)

# Prédiction
y_pred_rf = pipeline_rf.predict(X_test)

# Évaluation
print(classification_report(y_test, y_pred_rf))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.98      0.65      0.78      2145

    accuracy                           1.00    555719
   macro avg       0.99      0.83      0.89    555719
weighted avg       1.00      1.00      1.00    555719



In [16]:
# Calcul du ratio pour compenser le déséquilibre
scale = (y_train == 0).sum() / (y_train == 1).sum()

pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric="logloss", scale_pos_weight=scale))
])

pipeline_xgb.fit(X_train, y_train)

# Prédiction
y_pred_xgb = pipeline_xgb.predict(X_test)

# Évaluation
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00    553574
           1       0.36      0.92      0.52      2145

    accuracy                           0.99    555719
   macro avg       0.68      0.96      0.76    555719
weighted avg       1.00      0.99      0.99    555719



J'ai décidé d'utiliser le random forest pour etre certain que quand on detecte une fraude, c'est bien une fraud, XGboost tres fort mais beaucoup de faux positifs